# Vulkan Vacuum Grasping Lab
### *Adaptive In-Orbit Servicing & Geometric Intelligence*
---
## Pipeline Overview
1.  **Ingest** complex STL/PLY/PCD geometries.
2.  **Normalize** data (Unit scaling & Normal estimation).
3.  **Align** the object based on solid topological surfaces (RANSAC + DBSCAN).
4.  **Simulate** vacuum seal physics using the **Grasp Stability Score (GSS)**.
5.  **Identify** optimal contact points for robotic servicing.

## 🛠️ 1. Environment Setup
Initialize core geometric utilities and custom grasping modules.

In [11]:
import os
from os.path import join as pjoin
import numpy as np
import open3d as o3d
import yaml
import shutil
import json
from copy import deepcopy
import matplotlib.pyplot as plt

# Custom Framework Modules
from src.grippers.vacuum_gripper_v2 import VacuumGripper, VacuumGripperConfig
from src.grasping.vacuum_sampler_v2 import VacuumGraspSampler, VacuumSamplerConfig
import src.utils.geometry_utils as gu
from src.generic_geometry import GenericGeometry

print("✅ Framework Ready.")

✅ Framework Ready.


## 2. Selection Library
Configure your experiment by selecting a target part and a suction cup array. This block replaces messy path strings with an organized collection.

In [30]:
# === [COLLECTION] Gripper Parameters ===
GRIPPER_COLLECTION = {
    "single_small": "single_circle_cup_1cm_diam.yaml",
    "single_large": "single_circle_cup_2cm_diam.yaml",
    "double_standard": "double_cup_1cm_1cm.yaml",
    "double_large": "double_cup.yaml",
    "quad_array": "quadruple_cup_1cm_1cm.yaml",
    "franka_vacuum": "franka_vacuum.yaml"
}

# === [COLLECTION] Vulkan Part Library ===
VULKAN_PARTS = {
    "part_1": r"Test_part/Vulkan/stl/Vulkan part 1.stl",
    "part_2": r"Test_part/Vulkan/stl/Vulkan part 2.stl",
    "part_3": r"Test_part/Vulkan/stl/Vulkan part 3.stl",
    "part_4_body": r"Test_part/Vulkan/stl/Vulkan part 4-Body.stl",
    "nut_m12": r"Test_part/Vulkan/stl/Vulkan M12 Nut.stl",
    "subassembly": r"Test_part/Vulkan/stl/Vulkan subassembly JMS paper.stl",
    "subassembly_new": r"Test_part/Vulkan/stl/Vulkan subassembly JMS paper.stl",
    "complete_assembly": r"Test_part/Vulkan/stl/Vulkan complete assembly.stl",
    "screw": r"Test_part/Vulkan/stl/Vulkan screw.stl",
    "gcode_based_model": r"Test_part/Vulkan/stl/gcode based model.ply",
    "cubesat": r"Test_part/cubesat.stl"
}

# --- CHANGE YOUR SELECTION HERE ---
selected_gripper = "single_small"
selected_part = "gcode_based_model"
# ----------------------------------

gripper_path = pjoin("gripper_parameter", GRIPPER_COLLECTION[selected_gripper])
object_path = VULKAN_PARTS[selected_part]

gripper = VacuumGripper(gripper_path)
pcd = GenericGeometry(object_path)

print(f" Gripper Initialized: {gripper.config.name}")
print(f" Object Loaded: {os.path.basename(object_path)}")

[Open3D WARNING] geometry::TriangleMesh appears to be a geometry::PointCloud (only contains vertices, but no triangles).
 Gripper Initialized: Double_Schmalz_ECG
 Object Loaded: gcode based model.ply


#### Folder to save results

In [13]:
ATTEMPT_ID = "final_test_30_06_2026"
ATTEMPT_NAME = "_".join([selected_gripper, selected_part, ATTEMPT_ID])
RESULT_PATH = pjoin("results", ATTEMPT_NAME)
os.makedirs(RESULT_PATH, exist_ok=True)

##  3. Pre-Visualization
Check the initial spatial relationship. Use this to verify the object was loaded correctly.

In [32]:
print("Opening 3D Preview...")
o3d.visualization.draw_geometries(
    [gripper.collision_geometry.geometry, pcd.geometry], 
    window_name="Initial Setup Preview",
    point_show_normal=True
)

Opening 3D Preview...


## 4. Geometric Normalization
Convert to **meters**, downsample to a workable resolution, and remove outliers. 
> **Voxel Size**: defines the 'grain' of the surface.

In [31]:
report = pcd.get_dimensions_report()
report
#for validation of the gcode based pointcloud skip teh next steps and use this instead:
#pcd_down = pcd
#voxel_size = report["suggested_voxel_size"]/2


{'extents_xyz': array([0.0645, 0.101 , 0.0352]),
 'diagonal': 0.1249,
 'likely_unit': 'meters',
 'suggested_voxel_size': 0.00125,
 'details': 'Type: Open3D PointCloud. Size: 0.06 x 0.10 x 0.04. Unit: METERS.'}

#### Manual Evaluation:
The report gives a heuristic/notion if the scale is in millimeters or meters. If the "likely_unit" is "millimeters", we apply a scaling factor of 0.001 to convert to meters. This is crucial for ensuring that all subsequent geometric computations are accurate and consistent with the gripper's dimensions, which are typically defined in meters.

#### **However** the object may already be in meters, so it's important to make a manual check here

In [16]:
# ONLY RUN IF YOU WANT TO SCALE IT 1000X SMALLER (MM -> M for ex)
pcd.scale(0.001)
report = pcd.get_dimensions_report()
report

{'extents_xyz': array([0.071 , 0.1017, 0.0732]),
 'diagonal': 0.144,
 'likely_unit': 'meters',
 'suggested_voxel_size': 0.00144,
 'details': 'Type: Open3D Mesh. Size: 0.07 x 0.10 x 0.07. Unit: METERS.'}

I normally used the suggested voxel size of the dimension report rounded down (or a standard value like 0.0005).

In [33]:
# 2. Intelligent Downsampling
voxel_size = report["suggested_voxel_size"]/2 
pcd_down = pcd.downsample(voxel_size=voxel_size)
pcd_down = GenericGeometry(geometry=pcd_down)

# 3. Noise Reduction
# (OPTIONAL)
# pcd_down.remove_outliers_physical(inplace=True, min_neighbors=5)

print("--- Preprocessing Complete ---")
print(pcd_down.get_dimensions_report()['details'])
pcd_down.visualize()

--- Preprocessing Complete ---
Type: Open3D PointCloud. Size: 0.06 x 0.10 x 0.04. Unit: METERS.


In [34]:
pcd_down.visualize()

##  5. Intelligent Plane Alignment
We use **Hybrid RANSAC + DBSCAN** to find the top K solid surfaces. 
### **NOTE**: This should "ideally" be used for files which aren't already oriented (like the vulkan ones). But the cube-sat, for example, should run without this step to respect the normal z-axis (crucial for verticality evaluation).
- `k=3`: Generates 3 candidate part orientations.
- `theta_min_diff=30`: Ensures orientations are significantly different from each other.

In [42]:
pcd_aligned_list = gu.align_largest_plane_to_z(
    pcd_down.geometry, 
    distance_threshold=1.5 * voxel_size, 
    k=3, 
    theta_min_diff=30.0
)

print(f"Successfully identified {len(pcd_aligned_list)} primary alignment planes.")

for i, geom in enumerate(pcd_aligned_list):
    print(f"Viewing Aligned Geometry {i+1}...")
    GenericGeometry(geometry=geom).visualize()

Successfully identified 3 primary alignment planes.
Viewing Aligned Geometry 1...
Viewing Aligned Geometry 2...
Viewing Aligned Geometry 3...


In [8]:
# If you decide not to ru the above cell, run this one for compatibility with the rest of the workflow
pcd_aligned_list = [pcd_down.geometry]

## 6. Grasp Sampling Engine
Initialize the sampler using `config.yaml` and execute the discovery pipeline across all identified orientations.

In [20]:
pcd_down.visualize()

In [21]:
# Load Sampling Parameters
config_path = pjoin("config", "config.yaml")
with open(config_path, 'r') as f:
    raw_config = yaml.safe_load(f)

result_config_path = pjoin(RESULT_PATH, "config.yaml")
if not os.path.exists(result_config_path):
    shutil.copyfile(config_path, result_config_path)
        

In [22]:
sampler_config = VacuumSamplerConfig(**raw_config)

def mark_grasp_with_idx(grasp, idx):
    grasp.origin_idx = idx
    return grasp
    
all_candidates = []
geometry_sampler_points_list = []
print("Starting Batch Sampling...")
for i, aligned_geom in enumerate(pcd_aligned_list):
    sampler = VacuumGraspSampler(gripper, sampler_config)
    print(f"--- Processing Plane {i+1}/{len(pcd_aligned_list)} ---")
    found_grasps = sampler.sample_grasps(aligned_geom)
    found_grasps.sort(key=lambda x: x.score, reverse=True)
    found_grasps = list(map(lambda g: mark_grasp_with_idx(g,i), found_grasps))
    all_candidates.extend(found_grasps)
    geometry_sampler_points_list.append((aligned_geom, sampler, found_grasps))

# Global Ranking
all_candidates.sort(key=lambda x: x.score, reverse=True)
print(f"\n✨ Discovery Complete: Found {len(all_candidates)} valid candidates.")

Starting Batch Sampling...
--- Processing Plane 1/3 ---
[Debug] Visualizing Raycasting Mesh (Type: poisson)...
--- Processing Plane 2/3 ---
[Debug] Visualizing Raycasting Mesh (Type: poisson)...
--- Processing Plane 3/3 ---
[Debug] Visualizing Raycasting Mesh (Type: poisson)...

✨ Discovery Complete: Found 482 valid candidates.


#### Saving result into Json

In [23]:
cand = all_candidates[0]
d = {**cand.score_details, "score": cand.score,
     "position": cand.transform[:3,3].tolist(),
     "direction": cand.approach_vector.tolist()
    }
with open(pjoin(RESULT_PATH, "results.txt"), "a", encoding="utf-8") as f:
    f.write("\n")  # Add newline for readability if appending multiple records
    json.dump(d, f, ensure_ascii=False)
    f.write("\n")  # Add newline for readability if appending multiple records

## 📊 7. Visualization & Analytics
Gain insights into the grasp quality using spatial heatmaps and detailed pad projections.

### Choose the index of the orientation you want to analyze (if you have multiple aligned geometries from step 5)

In [24]:
DESIRED_IDX = 0

In [25]:
curr_geom, sampler, candidates = geometry_sampler_points_list[DESIRED_IDX]
curr_pcd = GenericGeometry(geometry=curr_geom)

In [59]:
candidates[0]

GraspCandidate(transform=array([[ 9.99999055e-01,  1.83150056e-07,  1.37478962e-03,
        -1.26688016e-02],
       [ 1.83150056e-07,  9.99999965e-01, -2.66440717e-04,
         8.63830425e-04],
       [-1.37478962e-03,  2.66440717e-04,  9.99999019e-01,
         6.86479326e-03],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         1.00000000e+00]]), score=np.float64(0.8093778050924698), contact_point=array([-0.0126688 ,  0.00086383,  0.00686479]), approach_vector=array([-1.37478965e-03,  2.66440723e-04, -9.99999041e-01]), score_details={'seal_score': np.float64(0.5955553330149383), 'verticality': np.float64(0.9986776660214246), 'torque': np.float64(0.837987518099143), 'raw_angle_deg': np.float64(0.07934003871452144), 'pad_scores': {'left_cup': np.float64(0.5955553330149383)}, 'failure_reason': []})

In [60]:
failures = [c.score_details['failure_reason'] for c in candidates]

In [61]:
len(sampler.valid_candidates)

175

In [62]:
len([f for f in failures if 'bad_seal' in f])

106

In [63]:
candidates[0].transform[:-1,3]

array([-0.0126688 ,  0.00086383,  0.00686479])

### Heatmaps of Scores

In [26]:
# === [ANALYSIS] Score Heatmap ===
# Attributes: 'total', 'seal_score', 'torque', 'verticality'
sampler.visualize_candidates_heatmap(curr_geom, attribute="total", relative_scale=False)

In [67]:
sampler.visualize_candidates_heatmap(curr_geom, attribute="verticality", relative_scale=False)

In [68]:
sampler.visualize_candidates_heatmap(curr_geom, attribute="torque", relative_scale=False)

In [69]:
sampler.visualize_candidates_heatmap(curr_geom, attribute="seal_score", relative_scale=False)

In [70]:
if candidates:
    best_candidate = candidates[0]
    sampler.visualize_grasp(curr_geom, best_candidate, show_safety_volume=False, show_frame=False)
    print(best_candidate.score_details)

{'seal_score': np.float64(0.5955553330149383), 'verticality': np.float64(0.9986776660214246), 'torque': np.float64(0.837987518099143), 'raw_angle_deg': np.float64(0.07934003871452144), 'pad_scores': {'left_cup': np.float64(0.5955553330149383)}, 'failure_reason': []}


In [15]:
if candidates:
    for k in range(10):
        best_candidate = candidates[k]
        sampler.visualize_grasp(curr_geom, best_candidate, show_safety_volume=False, show_frame=False)
        print(best_candidate.score_details)

{'seal_score': np.float64(0.61562102974242), 'verticality': np.float64(0.6), 'torque': np.float64(0.9333779260877119), 'raw_angle_deg': np.float64(0.0), 'pad_scores': {'left_cup': np.float64(0.61562102974242)}, 'failure_reason': []}
{'seal_score': np.float64(0.61562102974242), 'verticality': np.float64(0.6), 'torque': np.float64(0.8479395623801576), 'raw_angle_deg': np.float64(0.0), 'pad_scores': {'left_cup': np.float64(0.61562102974242)}, 'failure_reason': []}
{'seal_score': np.float64(0.61562102974242), 'verticality': np.float64(0.6), 'torque': np.float64(0.7916046427767549), 'raw_angle_deg': np.float64(0.0), 'pad_scores': {'left_cup': np.float64(0.61562102974242)}, 'failure_reason': []}
{'seal_score': np.float64(0.5130175247853499), 'verticality': np.float64(0.6), 'torque': np.float64(0.8082372308567632), 'raw_angle_deg': np.float64(0.0), 'pad_scores': {'left_cup': np.float64(0.5130175247853499)}, 'failure_reason': []}
{'seal_score': np.float64(0.5130175247853499), 'verticality': np

### Best Candidate (Overall)

In [71]:
# # === [ANALYSIS] Winner Inspection ===
if all_candidates:
    top_grasp = all_candidates[0]
    idx = top_grasp.origin_idx
    _geom, _sampler, _cands = geometry_sampler_points_list[idx]
    print(f"🏆 BEST GRASP DETAILS: Index {idx}")
    print(f"Global Score: {top_grasp.score:.4f}")
    print("----------------------")
    for key, val in top_grasp.score_details.items():
        if key != 'pad_scores':
            print(f"{key.replace('_',' ').title()}: {val}")
            
    # Visualize with 'Collision Shield' enabled (Blue cylinders showing safety volume)
    _sampler.visualize_grasp(_geom, top_grasp, show_safety_volume=True, show_frame=False)
else:
    print("❌ No grasps found. Consider lowering 'min_score' or increasing 'max_angle_deg' in config.yaml")

🏆 BEST GRASP DETAILS: Index 2
Global Score: 0.8136
----------------------
Seal Score: 0.521110916388071
Verticality: 0.9832501414959872
Torque: 0.9568646681475979
Raw Angle Deg: 1.0049915102407658
Failure Reason: []


## 🔍 8. Deep Physics Debugging
Visualize exactly how the suction pads are projected onto the object surface for specific candidates. This explains why a grasp might have a low Sealing Score.

In [21]:
# Select a candidate by index (0 is the best)
CANDIDATE_TO_DEBUG = 0

if CANDIDATE_TO_DEBUG < len(all_candidates):
    target = all_candidates[CANDIDATE_TO_DEBUG]
    print(f"🔍 Inspecting Candidate #{CANDIDATE_TO_DEBUG}...")
    sampler.debug_specific_grasp(pcd_down.geometry, target)
else:
    print("Index out of range.")

🔍 Inspecting Candidate #0...
--- Debugging Grasp at [0.05776431 0.09250446 0.05713594] ---
[Debug Sealing] Expected: 0.0 | Valid: 3 | Spacing: 1.0000
[Debug Sealing] Expected: 0.0 | Valid: 4 | Spacing: 1.0000
[Debug Sealing] Expected: 0.0 | Valid: 4 | Spacing: 1.0000
[Debug Sealing] Expected: 0.0 | Valid: 3 | Spacing: 1.0000
[Debug Sealing] Expected: 0.0 | Valid: 4 | Spacing: 1.0000
[Debug Sealing] Expected: 0.0 | Valid: 3 | Spacing: 1.0000
[Debug Sealing] Expected: 0.0 | Valid: 3 | Spacing: 1.0000
[Debug Sealing] Expected: 0.0 | Valid: 3 | Spacing: 1.0000


# ALign GCODE and STEP for subassembly
if you run directly this cells, wont work since I made many back and forth iterations to get the results

In [3]:
pcd_gcode = GenericGeometry(VULKAN_PARTS["gcode_based_model"])
pcd_step = GenericGeometry(VULKAN_PARTS["subassembly_new"])
pcd_step.scale(0.001)

[Open3D WARNING] geometry::TriangleMesh appears to be a geometry::PointCloud (only contains vertices, but no triangles).


In [29]:
pcd_step = GenericGeometry(geometry=curr_geom)

In [39]:
pcd_gcode = GenericGeometry(geometry=pcd_aligned_list[0])

In [41]:
pcd_gcode.visualize()
pcd_step.visualize()

In [6]:
pcd_gcode, pcd_step = pcd_gcode.downsample(voxel_size=0.0005), pcd_step.downsample(voxel_size=0.0005)

In [79]:
from src.utils.geometry_utils import align_pointclouds_icp

ImportError: cannot import name 'align_pointclouds_icp' from 'src.utils.geometry_utils' (/home/vpspepe/Documents/TUD/HiWi/PLCM/robot-grasping-victor/src/utils/geometry_utils.py)

In [8]:
pcd_gcode, pcd_step = GenericGeometry(geometry=pcd_gcode), GenericGeometry(geometry=pcd_step)

In [43]:
T = gu.align_point_clouds(pcd_gcode, pcd_step)

[GeometryUtils] Automatically estimated voxel size: 0.003749
[GeometryUtils] Running RANSAC-based global registration...
[Open3D WARNING] Too few correspondences (93) after mutual filter, fall back to original correspondences.
[GeometryUtils] RANSAC registration finished. Fitness: 0.9939, RMSE: 0.002304
[GeometryUtils] Running ICP local refinement (Point-to-Plane)...
[GeometryUtils] ICP registration finished. Fitness: 0.9394, RMSE: 0.000793


In [46]:
pcd_gcode.geometry.transform(T)

PointCloud with 17666 points.

In [50]:
pcd_gcode.visualize()

In [48]:
T_inv = np.linalg.inv(T)

In [49]:
pcd_gcode.geometry.transform(T_inv)

PointCloud with 17666 points.

In [51]:
pcd_step.geometry.transform(T)

PointCloud with 38575 points.

In [52]:
pcd_step.visualize()